# SAGE / DisjunctiveNet — Kaggle reproducibility runner

This notebook reproduces the certified Julia baseline and runs the Phase 9 native-disjunction cooling-control pilot by default. Phase 9 is an honest paper-spec reconstruction because the public upstream repository omits the experiment's numeric configuration.

Kaggle setup: turn **Internet on**. A GPU is not needed; the current solver path is CPU-bound. Each stage fails loudly if its acceptance token is absent.

The default `RUN_MODE = 'phase9'` runs the five-seed cooling-control pilot. Use `alpha` for the pre-registered geometry-aware mechanism gate, `phase8` for the Iris pilot, `phase7` for the completed scaling gate, `phase6` for the earlier matched-budget gate, or `full` for all accepted gates and the regression suite. Earlier negative runs remain documented rather than overwritten.

In [ ]:
from pathlib import Path
import json, os, platform, re, shutil, subprocess, sys, tarfile, time, urllib.request

REPO_URL = 'https://github.com/ghugn/SAGE_ICML2027.git'
REPO_BRANCH = 'main'
JULIA_VERSION = '1.13.3'  # matches the committed Manifest.toml and local verification
RUN_MODE = 'phase9'  # 'phase9': cooling pilot; 'alpha': novelty; 'phase8': Iris; 'phase7': scaling; 'phase6': earlier gate; 'full': all
assert RUN_MODE in ('phase6', 'phase7', 'phase8', 'alpha', 'phase9', 'full'), RUN_MODE
RUN_PRIOR_GATES = RUN_MODE == 'full'
RUN_PHASE6 = RUN_MODE in ('phase6', 'full')
RUN_PHASE7 = RUN_MODE in ('phase7', 'full')
RUN_PHASE8 = RUN_MODE in ('phase8', 'full')
RUN_ALPHA = RUN_MODE in ('alpha', 'full')
RUN_PHASE9 = RUN_MODE in ('phase9', 'full')
RUN_FULL_TESTS = RUN_MODE == 'full'

ON_KAGGLE = Path('/kaggle/working').exists()
WORK = Path('/kaggle/working') if ON_KAGGLE else Path.cwd()
DEPOT = WORK / 'julia_depot'
RUN_ID = time.strftime('%Y%m%d_%H%M%S') + f'_{RUN_MODE}'
LOG_DIR = WORK / 'sage_reproduction_logs' / RUN_ID
DEPOT.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
STAGE_RESULTS = {}
memory_kib = None
if Path('/proc/meminfo').exists():
    match = re.search(r'^MemTotal:\s+(\d+)', Path('/proc/meminfo').read_text(), re.MULTILINE)
    memory_kib = int(match.group(1)) if match else None
environment = {
    'work': str(WORK), 'on_kaggle': ON_KAGGLE, 'python': sys.version.split()[0],
    'platform': platform.platform(), 'machine': platform.machine(),
    'cpu_count': os.cpu_count(), 'memory_kib': memory_kib,
}
(LOG_DIR / 'environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
print(environment)
print('Run mode:', RUN_MODE)

## 1. Install or locate Julia
The archive is downloaded from Julia's official S3 distribution if the executable is not already available.

In [ ]:
def detected_julia_version(executable):
    try:
        return subprocess.check_output([str(executable), '--startup-file=no', '-e', 'print(VERSION)'], text=True).strip()
    except (OSError, subprocess.CalledProcessError):
        return None

def locate_or_install_julia(version=JULIA_VERSION):
    existing = shutil.which('julia')
    if existing and detected_julia_version(existing) == version:
        return Path(existing)
    if existing:
        print(f'Ignoring system Julia {detected_julia_version(existing)} at {existing}; reproducibility requires {version}')
    install_root = WORK / f'julia-{version}'
    executable = install_root / 'bin' / 'julia'
    if executable.exists() and detected_julia_version(executable) == version:
        return executable
    minor = '.'.join(version.split('.')[:2])
    archive = WORK / f'julia-{version}-linux-x86_64.tar.gz'
    url = f'https://julialang-s3.julialang.org/bin/linux/x64/{minor}/julia-{version}-linux-x86_64.tar.gz'
    print('Downloading', url)
    urllib.request.urlretrieve(url, archive)
    with tarfile.open(archive, 'r:gz') as bundle:
        bundle.extractall(WORK)
    assert executable.exists(), f'Julia executable not found at {executable}'
    return executable

JULIA = locate_or_install_julia()
print('Julia:', JULIA)
actual_julia_version = detected_julia_version(JULIA)
print('Julia version:', actual_julia_version)
assert actual_julia_version == JULIA_VERSION, (actual_julia_version, JULIA_VERSION)

## 2. Get the exact repository
When the notebook is run from a checked-out repository it uses that checkout. Otherwise it clones `main` into Kaggle's writable directory.

In [ ]:
if (Path.cwd() / 'Project.toml').exists():
    REPO = Path.cwd().resolve()
else:
    REPO = WORK / 'SAGE_ICML2027'
    if not (REPO / '.git').exists():
        subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1', REPO_URL, str(REPO)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', REPO_BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO), 'checkout', REPO_BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO), 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True)
print('Repository:', REPO)
subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], check=True)

## 3. Instantiate the Julia environment
The first run can take several minutes because Julia compiles packages. Every run receives a separate timestamped directory under `/kaggle/working/sage_reproduction_logs`, preventing stale logs from being mixed into a new evidence bundle.

In [ ]:
BASE_ENV = os.environ.copy()
BASE_ENV['JULIA_DEPOT_PATH'] = str(DEPOT)
BASE_ENV['JULIA_NUM_THREADS'] = str(max(1, os.cpu_count() or 1))

def run_stage(name, args, required_token=None):
    command = [str(JULIA), '--project=.', *args]
    print('\n$', ' '.join(command), flush=True)
    started = time.perf_counter()
    process = subprocess.Popen(command, cwd=REPO, env=BASE_ENV, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    lines = []
    for line in process.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    code = process.wait()
    elapsed = time.perf_counter() - started
    output = ''.join(lines)
    (LOG_DIR / f'{name}.log').write_text(output, encoding='utf-8')
    token_ok = required_token is None or required_token in output
    STAGE_RESULTS[name] = {'exit_code': code, 'seconds': round(elapsed, 3), 'required_token': required_token, 'token_ok': token_ok}
    (LOG_DIR / 'stage_results.json').write_text(json.dumps(STAGE_RESULTS, indent=2), encoding='utf-8')
    shutil.make_archive(str(WORK / 'sage_reproduction_bundle'), 'zip', root_dir=LOG_DIR)
    print(f'[{name}] exit={code} elapsed={elapsed:.3f}s token_ok={token_ok}', flush=True)
    if code != 0:
        raise RuntimeError(f'{name} failed with exit code {code}')
    if not token_ok:
        raise AssertionError(f'{name} did not emit {required_token!r}')
    return output

instantiate_log = run_stage('00_instantiate', ['-e', 'using Pkg; Pkg.instantiate(); Pkg.precompile()'])

## 4. Foundation and estimator verification
Runs only in `full` mode. These gates are reported as `SKIPPED` in single-phase resume modes.

In [ ]:
if RUN_PRIOR_GATES:
    foundation_log = run_stage('01_foundations', ['scripts/verify_foundations.jl'], 'FOUNDATION_VERIFICATION=PASS')
    sage_log = run_stage('02_sage_gradient', ['scripts/verify_sage_gradient.jl'], 'SAGE_GRADIENT_VERIFICATION=PASS')
    foundation_status = 'PASS'
    sage_status = 'PASS'
else:
    foundation_log = sage_log = ''
    foundation_status = sage_status = 'SKIPPED'
    print(f'Foundation and estimator gates skipped in {RUN_MODE} mode')

## 5. Controlled learning-through-discontinuity test
This gate runs only in `full` mode. Its detailed assumptions and limitations are in `docs/research_log/2026-11-13_phase5_discontinuity_learning.md`.

In [ ]:
if RUN_PRIOR_GATES:
    learning_log = run_stage('03_discontinuity_learning', ['scripts/benchmark_discontinuity_learning.jl'], 'DISCONTINUITY_LEARNING_GO_NO_GO=PASS')
    discontinuity_status = 'PASS'
else:
    learning_log = ''
    discontinuity_status = 'SKIPPED'
    print(f'Controlled discontinuity-learning gate skipped in {RUN_MODE} mode')

## 6. Multi-branch matched-budget benchmark
This proposal-aligned E1-E3 slice measures analytic/backend parity, gradient fidelity, certificate/pruning coverage, global solver calls, and complete backward wall time.

In [ ]:
if RUN_PHASE6:
    phase6_log = run_stage('04_multibranch_matched_budget', ['scripts/benchmark_multibranch_matched_budget.jl'], 'PHASE6_MATCHED_BUDGET=PASS')
    phase6_status = 'PASS'
else:
    phase6_log = ''
    phase6_status = 'SKIPPED'
    print('Phase 6 accepted gate skipped')

## 7. Phase 7 completed controlled scaling gate
This gate uses independent reference-convergence checks, controlled dimension/branch slices, common antithetic compact-`q` samples, exact feasible forwards, and repeated backend timing. It applies the proposal's matched-budget efficiency criterion: variance **or** total computation must improve. The earlier fixed-`N=64` smoke failure remains in the research log and is not relabeled.

In [ ]:
if RUN_PHASE7:
    BASE_ENV['SAGE_PHASE7_COMPLETE_OUTPUT'] = str(LOG_DIR / 'phase7_complete_results')
    phase7_log = run_stage('05_phase7_complete', ['scripts/benchmark_phase7_complete.jl'], 'PHASE7_COMPLETE=PASS')
    phase7_status = 'PASS'
else:
    phase7_log = ''
    phase7_status = 'SKIPPED'
    print('Phase 7 completed scaling gate skipped')

## 8. Phase 8 UCI Iris real-data pilot
This five-seed pilot trains the same small network with hard-local, STE, matched black-box smoothing and proposal-faithful SAGE. Species boxes and normalization are fit from each training split only; final forwards use certified exact search.

In [ ]:
if RUN_PHASE8:
    BASE_ENV['SAGE_PHASE8_OUTPUT'] = str(LOG_DIR / 'phase8_iris_results')
    phase8_log = run_stage('06_phase8_iris', ['scripts/benchmark_phase8_iris.jl'], 'PHASE8_IRIS_PILOT=PASS')
    phase8_status = 'PASS'
else:
    phase8_log = ''
    phase8_status = 'SKIPPED'
    print('Phase 8 Iris pilot skipped')

## 9. SAGE research-alpha novelty gate
This pre-registered gate validates the geometry proposal density, analytical gradient fidelity, exact feasibility and complete solver-work ablation. It compares against both independent and antithetic compact-density smoothing.

In [ ]:
if RUN_ALPHA:
    BASE_ENV['SAGE_ALPHA_OUTPUT'] = str(LOG_DIR / 'sage_alpha_novelty_results')
    alpha_log = run_stage('07_sage_alpha_novelty', ['scripts/benchmark_sage_novelty_alpha.jl'], 'SAGE_ALPHA_NOVELTY=PASS')
    alpha_status = 'PASS'
else:
    alpha_log = ''
    alpha_status = 'SKIPPED'
    print('SAGE research-alpha novelty gate skipped')

## 10. Phase 9 native-disjunction cooling-control pilot
This five-seed application pilot uses the paper-specified seven inputs, three controls, input-activated disjunctions, IID/OOD distributions and `7 -> 8 -> 8 -> 3` network. Missing upstream numeric constants are declared in `CoolingConfig`; this is a paper-spec reconstruction, not an exact reproduction.

In [ ]:
if RUN_PHASE9:
    BASE_ENV['SAGE_PHASE9_PROFILE'] = 'pilot'
    BASE_ENV['SAGE_PHASE9_OUTPUT'] = str(LOG_DIR / 'phase9_cooling_results')
    phase9_log = run_stage('08_phase9_cooling', ['scripts/benchmark_phase9_cooling.jl'], 'PHASE9_COOLING_PILOT=PASS')
    phase9_status = 'PASS'
else:
    phase9_log = ''
    phase9_status = 'SKIPPED'
    print('Phase 9 cooling-control pilot skipped')

## 11. Full regression suite
Runs only in `full` mode.

In [ ]:
if RUN_FULL_TESTS:
    tests_log = run_stage('09_full_tests', ['test/runtests.jl'], 'Test Summary:')
    full_tests_status = 'PASS'
else:
    full_tests_status = 'SKIPPED'
    print('Full tests skipped outside full mode')

## 12. Final acceptance summary

In [ ]:
summary = {
    'run_mode': RUN_MODE,
    'foundation': foundation_status,
    'sage_gradient': sage_status,
    'discontinuity_learning': discontinuity_status,
    'multibranch_matched_budget': phase6_status,
    'phase7_complete': phase7_status,
    'phase8_iris_pilot': phase8_status,
    'sage_alpha_novelty': alpha_status,
    'phase9_cooling_pilot': phase9_status,
    'full_tests': full_tests_status,
    'git_commit': subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip(),
    'environment': environment,
    'stages': STAGE_RESULTS,
    'logs': str(LOG_DIR),
}
(LOG_DIR / 'final_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
archive = shutil.make_archive(str(WORK / 'sage_reproduction_bundle'), 'zip', root_dir=LOG_DIR)
print(json.dumps(summary, indent=2))
print('Downloadable log bundle:', archive)
try:
    from IPython.display import FileLink, display
    display(FileLink(archive))
except ImportError:
    pass
if RUN_PHASE6:
    assert summary['multibranch_matched_budget'] == 'PASS'
if RUN_PHASE7:
    assert summary['phase7_complete'] == 'PASS'
if RUN_PHASE8:
    assert summary['phase8_iris_pilot'] == 'PASS'
if RUN_ALPHA:
    assert summary['sage_alpha_novelty'] == 'PASS'
if RUN_PHASE9:
    assert summary['phase9_cooling_pilot'] == 'PASS'
if RUN_PRIOR_GATES:
    assert all(summary[key] == 'PASS' for key in ('foundation', 'sage_gradient', 'discontinuity_learning'))
assert not RUN_FULL_TESTS or summary['full_tests'] == 'PASS'